# 人脸识别门禁系统

## 项目简介

本项目实现一个由树莓派负责视觉识别、Arduino 负责门锁执行和状态显示的人脸识别门禁系统。树莓派通过摄像头完成人脸检测和身份比对，再把识别结果通过串口发送给 Arduino，Arduino 根据识别结果控制继电器门锁、蜂鸣器和 LCD，并记录访问状态。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| 树莓派 | 1 | 运行人脸检测与识别程序 |
| 摄像头 | 1 | 采集访客图像 |
| Arduino Uno | 1 | 控制门锁、蜂鸣器和显示 |
| 继电器 / 电磁锁 | 1 | 门锁执行模块 |
| I2C LCD1602 | 1 | 状态显示 |
| 蜂鸣器、LED | 各 1 | 告警与授权提示 |


## 核心功能

- 树莓派侧执行人脸检测与身份识别。
- 已知用户授权开锁，陌生人拒绝并计次。
- 多次陌生人触发后进入锁定告警状态。
- LCD 轮播显示当前门禁状态、最后识别姓名和置信信息。
- 串口输出访问日志，便于上位机或串口助手保存记录。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| 继电器门锁 | D8 |
| 蜂鸣器 | D9 |
| 树莓派串口 RX / TX | D10、D11 |
| 状态 LED | D12 |
| LCD1602 SDA / SCL | I2C 总线 |


## 接线说明

- 树莓派串口和 Arduino 的 `SoftwareSerial` 口连接，注意 TX/RX 交叉并共地。
- 继电器控制端接 D8，用于驱动电磁锁或门禁执行回路。
- LCD1602 通过 I2C 接入 Arduino，地址默认为 `0x27`。
- 蜂鸣器接 D9，状态 LED 接 D12，用于不同门禁状态的本地反馈。
- 若使用电磁锁，门锁电源应独立设计，并通过继电器与逻辑电路隔离。


## 串口协议

- 树莓派向 Arduino 发送：`FACE,KNOWN,name,score`。
- 陌生人发送：`FACE,UNKNOWN,UNKNOWN,score`。
- 清空状态发送：`FACE,CLEAR,SYSTEM,0`。
- 例如：`FACE,KNOWN,Alice,92` 表示识别到已注册用户 Alice，匹配得分 92。
- Arduino 会在硬件串口输出 `LOG,GRANTED,...` 或 `LOG,DENIED,...` 作为访问日志。


## 运行方式

1. 上传 `src/face_recognition_access_control_system/face_recognition_access_control_system.ino` 到 Arduino。
2. 树莓派端部署摄像头采集、人脸比对和串口发送程序。
3. 先发送一组模拟帧，验证开锁、拒绝和锁定三种状态。
4. 根据实际门锁执行时间调整 `DOOR_UNLOCK_MS`。
5. 根据识别模型输出范围调整 `MIN_SCORE`，降低误识别开门风险。


## 控制逻辑说明

- Arduino 端维护 `IDLE / GRANTED / DENIED / LOCKDOWN` 四种访问状态。
- 已知用户且得分超过阈值时，门锁继电器拉高一段时间后自动恢复。
- 陌生人识别会按冷却时间累计次数，超过阈值则进入锁定告警状态。
- 授权与拒绝事件都会记录最后识别姓名、得分和事件类型，供屏幕与日志使用。
- 蜂鸣器根据拒绝和锁定状态使用不同节奏，便于区分普通拒绝和高风险告警。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 授权开门 | 已知用户识别成功后继电器能打开门锁 |
| 自动上锁 | 开门保持时间结束后会自动恢复锁定 |
| 陌生人拒绝 | 未知用户会触发拒绝提示且不解锁 |
| 连续陌生人告警 | 多次拒绝后进入锁定告警状态 |


## 可扩展方向

- 增加短信或微信通知推送。
- 增加本地 SD 卡访问日志保存。
- 增加管理员远程配置与黑名单机制。
- 将树莓派结果上传到 Web 后台形成完整门禁平台。
